# ComplaintIQ — QLoRA fine-tuning of Qwen2.5-1.5B-Instruct

Reproducible version of the cloud training workflow behind the delivered model (`Qwen2.5-1.5B-Instruct.Q4_K_M.gguf`, exported from Unsloth checkpoint-563). It trains on `data/finetuning/training.jsonl` — 9,000 real CFPB complaints across nine product/issue pairs — the same file uploaded to Unsloth for that run.

Runtime: Colab or Kaggle with a free NVIDIA GPU (T4 is enough for 1.5B in 4-bit). No paid jobs or API keys.

Reference: [Unsloth Qwen2.5 notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).

In [ ]:
%pip install -q unsloth
import torch
assert torch.cuda.is_available(), "A GPU runtime is required"
print(torch.cuda.get_device_name(0))

Mount Drive on Colab so checkpoints survive a disconnected session. On Kaggle, outputs go to `/kaggle/working`; Save Version to keep them.

In [ ]:
from pathlib import Path
import os
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/ComplaintIQ/qwen25')
else:
    OUTPUT = Path('/kaggle/working/complaintiq-output/qwen25')
# Change this to the extracted project directory.
PROJECT = Path('/content/complaintiq') if 'COLAB_RELEASE_TAG' in os.environ else Path('/kaggle/working/complaintiq')
os.chdir(PROJECT)
print(sum(1 for _ in open('data/finetuning/training.jsonl')), 'training examples')

**1. Smoke test** — two optimizer steps to verify loading, chat formatting, loss and saving.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/train_qlora.py', '--output', str(OUTPUT/'smoke'), '--steps', '2'], check=True)

**2. Full run** — one epoch over 9,000 examples (effective batch 16 ≈ 563 optimizer steps), resumable, then Q4_K_M GGUF export for the Mac.

In [ ]:
subprocess.run([sys.executable, 'scripts/train_qlora.py', '--output', str(OUTPUT/'full'), '--steps', '-1', '--epochs', '1', '--resume', '--export-gguf'], check=True)
print(sorted((OUTPUT/'full'/'gguf').glob('*.gguf')))

**3. Record the run** — loss history is in `run_status.json`; keep it with the adapter, GGUF and package versions.

In [ ]:
import json
status = json.loads((OUTPUT/'full'/'run_status.json').read_text())
print({k: status[k] for k in ('base_model', 'global_steps', 'train_loss')})
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=open(OUTPUT/'full'/'environment.txt', 'w'), check=True)

**4. Evaluate locally** — download the GGUF to the Mac, start `sh run_model.sh`, then run

```sh
.venv/bin/python -m scripts.evaluate_server                                   # original 153 reserved cases
.venv/bin/python -m scripts.evaluate_server --test-file data/finetuning/test.csv \
    --output artifacts/finetuned_evaluation_900.json                           # all 900 reserved cases
```

Never train on `validation.jsonl` or `test.jsonl`. Model weights are not committed to Git.